#Data Ingestion

#### Identify catalog and schema for use

In [0]:
%python

# list the target volumes
target_catalog_name = "databricks_simulated_retail_customer_data"
target_schema_name = "v02"
target_volume_names = ["business_daily_events", "customer_changes_daily", "subsidiary_daily_orders"]
target_path_name = f"/Volumes/{target_catalog_name}/{target_schema_name}"

# list the destination volumes
destination_catalog_name = "databricks_retail_data"
destination_schema_name = "v01"
destination_volume_names = ["business_daily_events", "customer_changes_daily", "subsidiary_daily_orders"]
destination_path_name = f"/Volumes/{destination_catalog_name}/{destination_schema_name}"

# create new catalog and schema in my destination
spark.sql(f"CREATE CATALOG IF NOT EXISTS {destination_catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {destination_schema_name}")

spark.sql(f"USE CATALOG {destination_catalog_name}")
spark.sql(f"USE SCHEMA {destination_schema_name}")

import pandas as pd
import glob
import os
from pathlib import Path

# create new volumes in organization
try:
    for target_vol, dest_vol in zip(target_volume_names, destination_volume_names):
        spark.sql(f"CREATE VOLUME IF NOT EXISTS {destination_catalog_name}.{destination_schema_name}.{dest_vol}")

        print(f"Target: ", target_vol)
        print(f"Destination: ", dest_vol)

        json_files = glob.glob(f"{target_path_name}/{target_vol}/**/*.json", recursive=True)
        csv_files = glob.glob(f"{target_path_name}/{target_vol}/**/*.csv", recursive=True)

        print(f"JSON Files: {json_files}")
        print(f"CSV Files: {csv_files}")

        for file in json_files:
            print(file)
            df = pd.read_json(file, lines=True)
            # print(f"Dataframe: {df}")
            df.to_json(f"{destination_path_name}/{dest_vol}/{Path(file).name}",
                orient="records", lines=True)
        for file in csv_files:
            print(file)
            df = pd.read_csv(file)
            # print(f"Dataframe: {df}")
            df.to_csv(f"{destination_path_name}/{dest_vol}/{Path(file).name}",
                index=False)

except Exception as e:
    print(f"Error writing files to destination: {e}")

In [0]:
CREATE TABLE IF NOT EXISTS customers_bronze AS
SELECT * FROM databricks_simulated_retail_customer_data.v01.customers;

CREATE TABLE IF NOT EXISTS sales_bronze AS
SELECT * FROM databricks_simulated_retail_customer_data.v01.sales;

CREATE TABLE IF NOT EXISTS sales_orders_bronze AS
SELECT * FROM databricks_simulated_retail_customer_data.v01.sales_orders;